# Knowledge Graph Extraction - Pre-Selected Texts

## Overview
This notebook extracts knowledge graphs from 4 pre-selected Northern Ireland education texts:
1. Madden (2011) CCEA Chp 3 - Changing Relationships (Option 2)
2. Madden (2011) CCEA Chp 2 - Peace, War, Neutrality (Option 1)
3. Updated Johnston&Johnston (Option 1)
4. Doherty (2001) - Northern Ireland since c.1960 (Option 2)

## Approach
- Uses **refined ontology** with separate definitions for Option 1 and Option 2
- **Enhanced extractor** with system/user message separation
- Validates entities and relationships against ontology
- Extracts discourse patterns and perspectives

In [1]:
# Cell 1: Imports and Setup
import os
import sys
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import warnings
import importlib
warnings.filterwarnings('ignore')

# Add scripts folder to path
sys.path.insert(0, '../scripts')

# Import our custom modules - with forced reload to pick up any changes
import refined_ontology
import enhanced_knowledge_extractor
import kg_utilities
importlib.reload(refined_ontology)
importlib.reload(kg_utilities)
importlib.reload(enhanced_knowledge_extractor)

from refined_ontology import RefinedOntology, CorpusOption, get_ontology
from enhanced_knowledge_extractor import NIKnowledgeExtractor, KnowledgeTriple
from kg_utilities import (
    SimpleRateLimiter,
    TextChunker,
    format_extraction_summary,
    load_target_documents,
    prepare_corpus_data,
    normalize_triples
)

# LLM imports
from openai import OpenAI

# Standard libraries
import json
import pickle
from datetime import datetime
from typing import List, Dict

print("✅ All imports successful! (modules force-reloaded)")
print(f"📅 Notebook run: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Verify ontology version
_test_ont = RefinedOntology()
print(f"   Option 1 relationship types: {len(_test_ont.get_relationships(CorpusOption.OPTION_1))}")
print(f"   Option 2 relationship types: {len(_test_ont.get_relationships(CorpusOption.OPTION_2))}")
print(f"   Entity aliases defined: {len(_test_ont.entity_aliases)}")



✅ All imports successful! (modules force-reloaded)
📅 Notebook run: 2026-03-03 07:43:14
   Option 1 relationship types: 123
   Option 2 relationship types: 130
   Entity aliases defined: 21


In [2]:
# Cell 2: Configuration

# Load environment variables
load_dotenv(override=True)

# Configuration
class Config:
    # API Keys
    OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
    
    # Model settings
    LLM_MODEL = 'gpt-4.1-mini-2025-04-14'
    
    # Chunking settings
    CHUNK_SIZE = 800  # Smaller for short texts
    CHUNK_OVERLAP = 150
    
    # Rate limiting
    API_DELAY = 1.0  # seconds between requests
    
    # Paths
    DATA_PATH = '../outputs/cleaned_text_data.csv'
    OUTPUT_PATH = '../graph_data'
    
    # Target files
    TARGET_FILES = [
        "Madden (2011) CCEA revision guide Chp 3. Changing Relationships.docx",
        "Madden (2011) ccea revision guide chp 2 peace war neutrality.docx",
        "Updated Johnston&Johnston no textboxes.docx",
        "Doherty (2001) Northern Ireland since c.1960 .docx"
    ]

config = Config()

# Initialize OpenAI client
if config.OPENAI_API_KEY:
    openai_client = OpenAI(api_key=config.OPENAI_API_KEY)
    print("✅ OpenAI client initialized")
    print(f"🎯 Using model: {config.LLM_MODEL}")
else:
    print("❌ ERROR: OPENAI_API_KEY not found in environment")
    print("💡 Set it in your .env file")

# Create output directory
os.makedirs(config.OUTPUT_PATH, exist_ok=True)
print(f"📁 Output directory: {config.OUTPUT_PATH}")

print("\n✅ Configuration complete!")

✅ OpenAI client initialized
🎯 Using model: gpt-4.1-mini-2025-04-14
📁 Output directory: ../graph_data

✅ Configuration complete!


In [3]:
# Cell 3: Load Data and Target Documents

print("📂 Loading dataset...")
df = pd.read_csv(config.DATA_PATH)
print(f"✅ Dataset loaded: {len(df):,} total rows")
print(f"   Columns: {df.columns.tolist()}")

# Load the 4 target documents (all chunks)
print("\n" + "="*70)
print("📚 Loading Target Documents")
print("="*70)

documents = load_target_documents(df, config.TARGET_FILES)

print("\n" + "="*70)
print("📊 Summary")
print("="*70)
print(f"Documents loaded: {len(documents)}/4")

# Show breakdown by religious group
option1_docs = [d for d in documents.values() if d['religious_group'] == 'option1']
option2_docs = [d for d in documents.values() if d['religious_group'] == 'option2']

print(f"\nOption 1 (Protestant/Unionist): {len(option1_docs)} documents")
for doc in option1_docs:
    print(f"  - {doc['filename'][:60]}...")
    print(f"    {doc['total_chunks']} chunks | {doc['total_words']:,} words")

print(f"\nOption 2 (Catholic/Nationalist): {len(option2_docs)} documents")
for doc in option2_docs:
    print(f"  - {doc['filename'][:60]}...")
    print(f"    {doc['total_chunks']} chunks | {doc['total_words']:,} words")

total_words = sum(d['total_words'] for d in documents.values())
print(f"\nTotal words across all documents: {total_words:,}")

📂 Loading dataset...
✅ Dataset loaded: 4,704 total rows
   Columns: ['filename', 'file_type', 'religious_group', 'content', 'content_length', 'word_count', 'chunk_id', 'chunk_count', 'has_url_content', 'has_ai_summary', 'source_url', 'publication_policy_year', 'cleaned_content']

📚 Loading Target Documents
âœ… Loaded: Madden (2011) CCEA revision guide Chp 3. Changing Relationships.docx
   Group: option2 | 77 chunks | 9,580 total words
âœ… Loaded: Madden (2011) ccea revision guide chp 2 peace war neutrality.docx
   Group: option1 | 75 chunks | 9,608 total words
âœ… Loaded: Updated Johnston&Johnston no textboxes.docx
   Group: option1 | 134 chunks | 22,452 total words
âœ… Loaded: Doherty (2001) Northern Ireland since c.1960 .docx
   Group: option2 | 3 chunks | 371 total words

📊 Summary
Documents loaded: 4/4

Option 1 (Protestant/Unionist): 2 documents
  - Madden (2011) ccea revision guide chp 2 peace war neutrality...
    75 chunks | 9,608 words
  - Updated Johnston&Johnston no textboxe

In [4]:
# Cell 4: Prepare Corpus Data (Already Chunked!)

print("📋 Organizing pre-chunked data by perspective...")
corpus_data = prepare_corpus_data(documents)

print("\n" + "="*70)
print("📊 Corpus Data Summary")
print("="*70)
print(f"Option 1 chunks: {len(corpus_data['option1']['chunks'])}")
print(f"Option 2 chunks: {len(corpus_data['option2']['chunks'])}")
print(f"Total chunks: {len(corpus_data['option1']['chunks']) + len(corpus_data['option2']['chunks'])}")

# Show sample chunks
print("\n📝 Sample chunks:")
if corpus_data['option1']['chunks']:
    sample = corpus_data['option1']['chunks'][0]
    print(f"\nOption 1 chunk example (first 200 chars):")
    print(f"  {sample[:200]}...")

if corpus_data['option2']['chunks']:
    sample = corpus_data['option2']['chunks'][0]
    print(f"\nOption 2 chunk example (first 200 chars):")
    print(f"  {sample[:200]}...")

📋 Organizing pre-chunked data by perspective...

ðŸ“„ Madden (2011) CCEA revision guide Chp 3. Changing Relationsh...
   Group: option2
   Chunks: 77

ðŸ“„ Madden (2011) ccea revision guide chp 2 peace war neutrality...
   Group: option1
   Chunks: 75

ðŸ“„ Updated Johnston&Johnston no textboxes.docx...
   Group: option1
   Chunks: 134

ðŸ“„ Doherty (2001) Northern Ireland since c.1960 .docx...
   Group: option2
   Chunks: 3

📊 Corpus Data Summary
Option 1 chunks: 209
Option 2 chunks: 80
Total chunks: 289

📝 Sample chunks:

Option 1 chunk example (first 200 chars):
  war relationship outbreak second world war september established government act act partitioned part governed separate southern later became irish free state act gave parliament control area health tr...

Option 2 chunk example (first 200 chars):
  relation republic emergence development established government act act partitioned county southern later became irish free state population reacted new arrangement unionist most

In [5]:
# Cell 5: Initialize Knowledge Extractor

print("🤖 Initializing Knowledge Extractor...")

# Initialize extractor
extractor = NIKnowledgeExtractor(
    client=openai_client,
    model=config.LLM_MODEL
)

# Initialize rate limiter
rate_limiter = SimpleRateLimiter(delay_seconds=config.API_DELAY)

print("\n✅ Extractor ready!")
print(f"   Model: {config.LLM_MODEL}")
print(f"   Rate limit: {config.API_DELAY}s between requests")
print("   Ontology: Auto-detected per corpus")

# Show ontology info
print("\n📚 Ontology Overview:")
ontology = RefinedOntology()

print("\nOption 1:")
opt1_labels = ontology.get_all_labels(CorpusOption.OPTION_1)
opt1_rels = ontology.get_relationships(CorpusOption.OPTION_1)
print(f"  Entity types: {len(opt1_labels)}")
print(f"  Relationship types: {len(opt1_rels)}")

print("\nOption 2:")
opt2_labels = ontology.get_all_labels(CorpusOption.OPTION_2)
opt2_rels = ontology.get_relationships(CorpusOption.OPTION_2)
print(f"  Entity types: {len(opt2_labels)}")
print(f"  Relationship types: {len(opt2_rels)}")

🤖 Initializing Knowledge Extractor...
Knowledge extractor initialized with model: gpt-4.1-mini-2025-04-14

✅ Extractor ready!
   Model: gpt-4.1-mini-2025-04-14
   Rate limit: 1.0s between requests
   Ontology: Auto-detected per corpus

📚 Ontology Overview:

Option 1:
  Entity types: 43
  Relationship types: 123

Option 2:
  Entity types: 46
  Relationship types: 130


In [6]:
# Cell 6: Extract Knowledge - Option 1 (Protestant/Unionist)

print("\n" + "="*70)
print("🔍 EXTRACTING KNOWLEDGE GRAPH - OPTION 1")
print("="*70)

start_time = datetime.now()

option1_triples = extractor.extract_from_corpus(
    chunks=corpus_data['option1']['chunks'],
    metadata=corpus_data['option1']['metadata'],
    corpus_name='option1',
    save_progress=True,
    rate_limiter=rate_limiter
)

elapsed_time = (datetime.now() - start_time).total_seconds()

print("\n✅ Option 1 extraction complete!")
print(f"   Time elapsed: {elapsed_time:.1f}s")
print(f"   Triples extracted: {len(option1_triples)}")

# Show sample triples
if option1_triples:
    print("\n📝 Sample triples:")
    for i, triple in enumerate(option1_triples[:3]):
        print(f"\n  {i+1}. {triple.subject} --[{triple.predicate}]--> {triple.object}")
        print(f"     Types: {triple.subject_type} --> {triple.object_type}")
        print(f"     Confidence: {triple.confidence:.2f}")


🔍 EXTRACTING KNOWLEDGE GRAPH - OPTION 1

Starting extraction from option1...
Total chunks: 209

ðŸ“š Ontology set for: OPTION_1
   Entity types: 43
   Relationship types: 123


Extracting option1:   0%|          | 1/209 [00:13<47:34, 13.73s/it]

  ⚠️  Invalid predicate: Keep
  ⚠️  Invalid predicate: Ensure


Extracting option1:   1%|          | 2/209 [00:26<45:11, 13.10s/it]

  ⚠️  Invalid predicate: Reach
  ⚠️  Invalid predicate: Delight


...

  ⚠️  Invalid predicate: Refer
  ⚠️  Invalid predicate: Common
  ⚠️  Invalid predicate: Worry
  ⚠️  Invalid predicate: Watch


Extracting option1: 100%|█████████▉| 208/209 [21:16<00:08,  8.39s/it]

  ⚠️  Invalid predicate: Bring
  ⚠️  Invalid predicate: Spend


Extracting option1: 100%|██████████| 209/209 [21:24<00:00,  6.15s/it]


✅ Extraction complete for option1!
  Extracted: 709 knowledge relations
  Success rate: 100.0%
  Total cost: $0.092

🛠️ Normalizing entity names...
  Entity dedup: merged 32 case-variant names
    ['war effort', 'War Effort', 'War effort'] -> 'War effort'
    ['parliament', 'Parliament'] -> 'Parliament'
    ['Unionist', 'unionist'] -> 'Unionist'
    ['Irish economy', 'Irish Economy'] -> 'Irish economy'
    ['Economic war', 'Economic War'] -> 'Economic War'
    ['War', 'war'] -> 'War'
    ['New constitution', 'new constitution'] -> 'New constitution'
    ['Government', 'government'] -> 'Government'
    ['prime minister', 'Prime Minister'] -> 'Prime Minister'
    ['fire engine', 'Fire Engine'] -> 'Fire Engine'
    ['Coventry', 'coventry'] -> 'Coventry'
    ['death', 'Death'] -> 'death'
    ['people', 'People'] -> 'People'
    ['Home Guard', 'Home guard'] -> 'Home Guard'
    ['Germany', 'germany'] -> 'Germany'
    ['Labour government', 'Labour Government'] -> 'Labour Government'
    ['ul

In [7]:
# Cell 7: Extract Knowledge - Option 2 (Catholic/Nationalist)

print("\n" + "="*70)
print("🔍 EXTRACTING KNOWLEDGE GRAPH - OPTION 2")
print("="*70)

start_time = datetime.now()

option2_triples = extractor.extract_from_corpus(
    chunks=corpus_data['option2']['chunks'],
    metadata=corpus_data['option2']['metadata'],
    corpus_name='option2',
    save_progress=True,
    rate_limiter=rate_limiter
)

elapsed_time = (datetime.now() - start_time).total_seconds()

print("\n✅ Option 2 extraction complete!")
print(f"   Time elapsed: {elapsed_time:.1f}s")
print(f"   Triples extracted: {len(option2_triples)}")

# Show sample triples
if option2_triples:
    print("\n📝 Sample triples:")
    for i, triple in enumerate(option2_triples[:3]):
        print(f"\n  {i+1}. {triple.subject} --[{triple.predicate}]--> {triple.object}")
        print(f"     Types: {triple.subject_type} --> {triple.object_type}")
        print(f"     Confidence: {triple.confidence:.2f}")


🔍 EXTRACTING KNOWLEDGE GRAPH - OPTION 2

Starting extraction from option2...
Total chunks: 80

ðŸ“š Ontology set for: OPTION_2
   Entity types: 46
   Relationship types: 130


Extracting option2:   0%|          | 0/80 [00:00<?, ?it/s]

  ðŸ“Š 210 requests | 1289s elapsed | 0.2 req/s


Extracting option2:   1%|▏         | 1/80 [00:07<09:57,  7.56s/it]

  ⚠️  Invalid predicate: Govern


...

  ⚠️  Invalid predicate: Launch
  ⚠️  Invalid predicate: Engage
  ⚠️  Invalid predicate: Seen

  💼 Progress: 75/80 chunks
     Extracted: 288 relations
     Cost so far: $0.126
  💼 Progress saved: ../graph_data/kg_triples_option2_progress_75_20260303_081357.pkl
  ðŸ“Š 285 requests | 1789s elapsed | 0.2 req/s


Extracting option2:  95%|█████████▌| 76/80 [08:26<00:27,  6.90s/it]

  ⚠️  Invalid predicate: Absence


Extracting option2: 100%|██████████| 80/80 [08:33<00:00,  6.42s/it]


✅ Extraction complete for option2!
  Extracted: 294 knowledge relations
  Success rate: 100.0%
  Total cost: $0.127

🛠️ Normalizing entity names...
  Entity dedup: merged 7 case-variant names
    ['Civil rights movement', 'Civil Rights Movement', 'civil rights movement'] -> 'Civil Rights Movement'
    ['Power sharing', 'Power Sharing'] -> 'Power Sharing'
    ['NICRA march', 'NICRA March'] -> 'NICRA March'
    ['violence', 'Violence'] -> 'Violence'
    ['British Government', 'British government'] -> 'British Government'
    ['End', 'end'] -> 'End'
✅ Entity names normalized
  💼 Progress saved: ../graph_data/kg_triples_option2_final_20260303_081411.pkl

📋 Invalid predicates report saved: ../graph_data/invalid_predicates_option2_20260303_081411.json
   Total invalid: 305
   Unique predicates: 213

   Top 10 invalid predicates:
      Try                  :   6 times
      Meet                 :   6 times
      Appear               :   5 times
      Pay                  :   4 times
      En

In [8]:
# Cell 8: Final Summary and Statistics

print("\n" + "="*70)
print("📊 FINAL EXTRACTION SUMMARY")
print("="*70)

# Combined statistics
total_triples = len(option1_triples) + len(option2_triples)

print(f"\nDocuments processed: {len(documents)}")
print(f"Total chunks: {len(corpus_data['option1']['chunks']) + len(corpus_data['option2']['chunks'])}")
print(f"\nOption 1 triples: {len(option1_triples)}")
print(f"Option 2 triples: {len(option2_triples)}")
print(f"Total triples: {total_triples}")

# Get rate limiter stats
rate_stats = rate_limiter.get_summary()
print(f"\nAPI calls: {rate_stats['total_requests']}")
print(f"Total cost: ${rate_stats['total_cost']:.3f}")
print(f"Time elapsed: {rate_stats['elapsed_seconds']}s")

# Show extractor stats
print("\n" + "="*70)
extractor.print_final_summary()

# Save combined results
print("\n💾 Saving results...")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

all_results = {
    'option1': option1_triples,
    'option2': option2_triples,
    'metadata': {
        'extraction_date': timestamp,
        'model': config.LLM_MODEL,
        'target_files': config.TARGET_FILES,
        'total_triples': total_triples,
        'statistics': extractor.extraction_stats
    }
}

output_file = os.path.join(config.OUTPUT_PATH, f'preselected_kg_{timestamp}.pkl')
with open(output_file, 'wb') as f:
    pickle.dump(all_results, f)

print(f"✅ Results saved: {output_file}")
print("\n🎉 Extraction complete!")


📊 FINAL EXTRACTION SUMMARY

Documents processed: 4
Total chunks: 289

Option 1 triples: 709
Option 2 triples: 294
Total triples: 1003

API calls: 289
Total cost: $0.127
Time elapsed: 1802s


⚠️ FINAL KNOWLEDGE GRAPH EXTRACTION SUMMARY
Total chunks processed:  289
Successful extractions:  289
Failed extractions:      0
Success rate:            100.0%
Total relations:         1,003
Avg relations/chunk:     3.5
API calls made:          289
Total cost:              $0.127

⚠️  VALIDATION STATISTICS
Invalid predicates found: 305
Unique invalid predicates: 213

Top 5 most common:
  Try                  :   6 times
  Meet                 :   6 times
  Appear               :   5 times
  Pay                  :   4 times
  Ensure               :   3 times

💾 Saving results...
✅ Results saved: ../graph_data/preselected_kg_20260303_081411.pkl

🎉 Extraction complete!


In [9]:
# Cell 9: Quick Analysis - Compare Perspectives

from collections import Counter

print("\n" + "="*70)
print("🔍 COMPARATIVE ANALYSIS")
print("="*70)

# Analyze relationships by perspective
print("\n📊 Top Relationship Types:")

option1_rels = [t.predicate for t in option1_triples]
option2_rels = [t.predicate for t in option2_triples]

print("\nOption 1 (Protestant/Unionist):")
for rel, count in Counter(option1_rels).most_common(10):
    print(f"  {rel:20s} : {count:3d}")

print("\nOption 2 (Catholic/Nationalist):")
for rel, count in Counter(option2_rels).most_common(10):
    print(f"  {rel:20s} : {count:3d}")

# Analyze entities
print("\n📊 Top Entities Mentioned:")

option1_entities = [t.subject for t in option1_triples] + [t.object for t in option1_triples]
option2_entities = [t.subject for t in option2_triples] + [t.object for t in option2_triples]

print("\nOption 1:")
for entity, count in Counter(option1_entities).most_common(10):
    print(f"  {entity:30s} : {count:3d}")

print("\nOption 2:")
for entity, count in Counter(option2_entities).most_common(10):
    print(f"  {entity:30s} : {count:3d}")

# Find unique vs shared entities
opt1_set = set(option1_entities)
opt2_set = set(option2_entities)
shared = opt1_set & opt2_set
opt1_only = opt1_set - opt2_set
opt2_only = opt2_set - opt1_set

print("\n📊 Entity Overlap:")
print(f"  Shared entities: {len(shared)}")
print(f"  Option 1 only: {len(opt1_only)}")
print(f"  Option 2 only: {len(opt2_only)}")

if shared:
    print(f"\n  Shared entities: {list(shared)[:10]}")


🔍 COMPARATIVE ANALYSIS

📊 Top Relationship Types:

Option 1 (Protestant/Unionist):
  Become               :  31
  Remain               :  30
  Want                 :  26
  Attack               :  25
  Cause                :  23
  Use                  :  22
  Fear                 :  22
  Declare              :  22
  Support              :  20
  React                :  16

Option 2 (Catholic/Nationalist):
  React                :  16
  Organise             :  15
  Result               :  11
  Oppose               :  11
  Resign               :  11
  Demand               :  10
  Support              :   9
  Introduce            :   7
  Increase             :   7
  Feel                 :   6

📊 Top Entities Mentioned:

Option 1:
  Éire                           :  78
  Éamon de Valera                :  72
  Britain                        :  47
  Neutrality                     :  34
  Unionist                       :  32
  Winston Churchill              :  31
  Irish Free State            

In [10]:
# Cell 10: Export for Neo4j or Further Analysis

print("\n" + "="*70)
print("💾 EXPORTING DATA")
print("="*70)

# Export to JSON for easy viewing
json_file = os.path.join(config.OUTPUT_PATH, f'preselected_kg_{timestamp}.json')

json_data = {
    'option1': [
        {
            'subject': t.subject,
            'predicate': t.predicate,
            'object': t.object,
            'subject_type': t.subject_type,
            'object_type': t.object_type,
            'confidence': t.confidence,
            'source_corpus': t.source_corpus
        }
        for t in option1_triples
    ],
    'option2': [
        {
            'subject': t.subject,
            'predicate': t.predicate,
            'object': t.object,
            'subject_type': t.subject_type,
            'object_type': t.object_type,
            'confidence': t.confidence,
            'source_corpus': t.source_corpus
        }
        for t in option2_triples
    ],
    'metadata': all_results['metadata']
}

with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(json_data, f, indent=2, ensure_ascii=False)

print(f"✅ JSON exported: {json_file}")

# Export to CSV for easy analysis in Excel/R
csv_file = os.path.join(config.OUTPUT_PATH, f'preselected_kg_{timestamp}.csv')

all_triples_list = []
for t in option1_triples + option2_triples:
    all_triples_list.append({
        'corpus': t.source_corpus,
        'subject': t.subject,
        'subject_type': t.subject_type,
        'predicate': t.predicate,
        'object': t.object,
        'object_type': t.object_type,
        'confidence': t.confidence,
        'chunk_id': t.chunk_id
    })

df_export = pd.DataFrame(all_triples_list)
df_export.to_csv(csv_file, index=False)

print(f"✅ CSV exported: {csv_file}")
print(f"\n📊 Exported {len(all_triples_list)} total triples")
print("\n🎉 All exports complete!")


💾 EXPORTING DATA
✅ JSON exported: ../graph_data/preselected_kg_20260303_081411.json
✅ CSV exported: ../graph_data/preselected_kg_20260303_081411.csv

📊 Exported 1003 total triples

🎉 All exports complete!


In [11]:
# Cell 11: Load Knowledge Graph into Neo4j (v2 - weighted edges)

# Import from the new module (make sure neo4j_loader_v2.py is in ../scripts/)
import importlib
import neo4j_loader_v2
importlib.reload(neo4j_loader_v2)

from neo4j_loader_v2 import (
    Neo4jConfig, connect_to_neo4j, clear_database, create_indexes,
    load_entities, load_relationships_weighted, verify_data, print_sample_queries
)

print("\n" + "="*70)
print("🔗 CONNECTING TO NEO4J")
print("="*70)

# ============================================================
# RE-NORMALIZE triples with the LATEST ontology before loading
# This catches any entity inconsistencies that slipped through
# extraction (e.g., "hitler" vs "Hitler", "Éamon De Valera" etc.)
# ============================================================
print("\n🔧 Re-normalizing triples with latest ontology...")
ontology = RefinedOntology()

option1_before = len(set(t.subject for t in option1_triples) | set(t.object for t in option1_triples))
option2_before = len(set(t.subject for t in option2_triples) | set(t.object for t in option2_triples))

option1_triples = normalize_triples(option1_triples, ontology)
option2_triples = normalize_triples(option2_triples, ontology)

option1_after = len(set(t.subject for t in option1_triples) | set(t.object for t in option1_triples))
option2_after = len(set(t.subject for t in option2_triples) | set(t.object for t in option2_triples))

print(f"   Option 1 entities: {option1_before} → {option1_after} (merged {option1_before - option1_after})")
print(f"   Option 2 entities: {option2_before} → {option2_after} (merged {option2_before - option2_after})")
print("✅ Re-normalization complete")

# ============================================================
# Connect and load into Neo4j
# ============================================================
neo4j_config = Neo4jConfig()
driver = connect_to_neo4j(neo4j_config)

if driver:
    try:
        # Clear existing data
        clear_database(driver, neo4j_config.DATABASE)
        
        # Create indexes
        create_indexes(driver, neo4j_config.DATABASE)
        
        # Load entities
        entities = load_entities(driver, neo4j_config.DATABASE, option1_triples, option2_triples)
        
        # Load relationships WITH EDGE MERGING (v2 change)
        load_relationships_weighted(driver, neo4j_config.DATABASE, option1_triples, option2_triples)
        
        # Verify
        verify_data(driver, neo4j_config.DATABASE)
        
        # Print sample queries
        print_sample_queries()
        
        print("\n" + "="*70)
        print("🎉 KNOWLEDGE GRAPH LOADED INTO NEO4J!")
        print("="*70)
        
    except Exception as e:
        print(f"\n❌ Error loading data: {e}")
        import traceback
        traceback.print_exc()
    
    finally:
        driver.close()
        print("\n✅ Neo4j connection closed")
else:
    print("\n❌ Cannot load data without Neo4j connection")
    print("   Make sure Neo4j is running and credentials are correct")




🔗 CONNECTING TO NEO4J

🔧 Re-normalizing triples with latest ontology...
   Option 1 entities: 531 → 531 (merged 0)
   Option 2 entities: 277 → 277 (merged 0)
✅ Re-normalization complete
✅ Connected to Neo4j at neo4j+s://a052fe1a.databases.neo4j.io

⚠️  Clearing existing data...
✅ Database cleared

📊 Creating indexes...
✅ Index on entity names created

📦 Loading entities...
   Found 774 unique entities
✅ Created 774 entity nodes

🔗 Loading relationships (with edge merging)...
   Raw edges: 1003
   After merging: 948 (removed 55 duplicates)
   Edges with weight > 1: 40
   Max edge weight: 7
✅ Created 948 merged relationships

🔍 Verifying data...
✅ Nodes in Neo4j: 774
✅ Merged relationships: 948
   (representing 1,003 original edges)

   By corpus:
      Protestant/Unionist       : 497 entities
      Catholic/Nationalist      : 243 entities
      Shared                    :  34 entities

   Edge weight stats:
      Edges with weight > 1: 40
      Max weight: 7
      Avg weight (of multi)